# **Langkah Praktikum**

K-1. Import Library dan Inisialisasi


In [ ]:
!pip install faker
import numpy as np
import pandas as pd
from faker import Faker
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DIR_KERJA = "/con tent/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_KERJA))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[]


K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)


In [ ]:
import pandas as pd
import numpy as np
import random
from faker import Faker

SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["pro", "Lite", "max", "basic", " "])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000])
    qty = random.randint(1, 5)

    # Variasi format harga
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "


    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "transaction_date": tanggal,
        "payment_method": metode,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


K-3. Deteksi dan Penanganan Missing Value


In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
transaction_date      0
payment_method       16
shipping_city        30
rating              132
dtype: int64


In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"]=df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_duplicates():", len(df))

Jumlah baris setelah drop_duplicates(): 495


K-4. Deteksi dan Penanganan Duplicate


In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated () .sum() )

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


K-5. Koreksi Tipe Data dan Standardisasi Format


a. Standardisasi teks kategorikal (category, payment_method, shipping_city):


In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik):


In [ ]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    X = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
      return float (x)
    except ValueError:
      return np.nan

df["price"] = df["price"].apply(bersihkan_harga)



c. Standardisasi format tanggal ke YYYY-MM-DD:

In [ ]:
def parse_tanggal (x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y") :
        try:
          return pd. to_datetime(x, format=fmt)
        except ValueError:
          continue
    return pd. NaT

df["transaction date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")



d. Finalisasi tipe data:


In [ ]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

K-6. Ekspor Dataset Bersih


In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

folder_drive = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_drive, exist_ok=True)   # buat folder jika belum ada

path_drive = folder_drive + "/transaksi_bersih.csv"
df.to_csv(path_drive, index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Studi kasus**

Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan bulanan: total transaksi yang
dilaporkan tim IT (515) tidak sama dengan total yang dipakai tim Finance (490). Sebagai calon data
engineer, jelaskan kepada tim Finance:



1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda
lakukan.




  Jawaban : Perbedaan ini terjadi karena tim IT menarik data mentah langsung dari sumbernya yang berjumlah 515 baris, sementara tim Finance menggunakan data yang sudah melewati tahap pembersihan (pra-pemrosesan) menjadi 490 baris. Selama proses pembersihan tersebut, sebanyak 25 baris dihapus secara sengaja karena kualitasnya buruk, yang secara spesifik terdiri dari 15 baris transaksi ganda (duplikat) dan 10 baris transaksi yang tidak memiliki informasi wajib seperti nama pelanggan atau metode pembayaran

2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep
Veracity

  Jawaban : Angka 490 baris "lebih benar" dan merupakan data valid yang seharusnya digunakan untuk pelaporan. Hal ini berkaitan langsung dengan dimensi Veracity dalam Big Data, yang mengukur seberapa akurat dan bisa dipercayanya suatu data untuk dijadikan dasar pengambilan keputusan. Jika kita mempertahankan 515 baris, laporan tim Finance akan ikut menghitung transaksi yang tercatat dua kali atau transaksi bodong tanpa pembayar, yang berisiko fatal pada akurasi laporan keuangan (sebuah kondisi garbage in, garbage out). Volume data yang sekadar besar tidak ada gunanya jika kualitas atau Veracity-nya buruk


3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing
value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?

  Jawaban : Rating sengaja dibiarkan kosong karena memberikan penilaian adalah tindakan opsional dari pihak pembeli. Memaksa mengisi kekosongan tersebut dengan angka tebakan atau asumsi (imputasi) justru akan mendistorsi dan merusak analisis di kemudian hari. Untuk menjawab kebutuhan tim Finance mengenai "rating rata-rata", perhitungannya cukup dilakukan dengan mengambil rata-rata murni dari transaksi yang memang benar-benar diberi rating oleh pembeli secara eksplisit, lalu menanganinya secara spesifik saat proses analisis tanpa melibatkan data kosong

  

# **Latihan**

1. Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris
transaksi_mentah.csv dan transaksi_bersih.csv dengan hasil SEED = 42. Apakah
jumlahnya sama? Jelaskan mengapa.


sama, Jumlah baris pada transaksi_mentah.csv dihasilkan dari penetapan variabel N = 500 yang kemudian ditambah secara eksplisit dengan baris duplikat sebanyak n=15. Skema ini memastikan bahwa dataset mentah akan selalu memiliki total awal sebanyak 515 baris, angka berapapun yang dimasukkan ke dalam konfigurasi SEED.


2. Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk
memeriksa apakah ada harga tidak valid

In [ ]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['is_valid_price'] = df['price'] > 0
df[df['is_valid_price'] == False]

,transaction_id,customer_name,product_name,category,price,quantity,transaction_date,payment_method,shipping_city,rating,transaction date,is_valid_price
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur pro,Kesehatan,NaN,1,15/09/2026,E-Wallet,Bau-Bau,4.0,2026-09-15,False
1,TRX00430,Faizah Kusumo,Voluptates max,Rumah Tangga,NaN,1,18-07-2026,COD,Cilegon,4.0,2026-07-18,False
3,TRX00121,"Ilsa Mahendra, S.T.",Quia basic,Kesehatan,NaN,2,17-08-2026,E-Wallet,Bengkulu,NaN,2026-08-17,False
10,TRX00296,R. Kayun Puspasari,Placeat pro,Olahraga,NaN,1,31-08-2026,Transfer Bank,Tomohon,NaN,2026-08-31,False
17,TRX00467,Ayu Prastuti,Nulla,Kesehatan,NaN,3,01-07-2026,COD,Pasuruan,1.0,2026-07-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...
491,TRX00429,Amalia Tarihoran,Placeat Lite,Olahraga,NaN,5,09/07/2026,Transfer Bank,Kota Administrasi Jakarta Utara,NaN,2026-07-09,False
498,TRX00393,Rahmi Ramadan,Possimus basic,Rumah Tangga,NaN,1,2026-09-17,Transfer Bank,Kota Administrasi Jakarta Barat,NaN,2026-09-17,False
506,TRX00186,Ir. Sabri Wijaya,Minima,Olahraga,NaN,1,14/07/2026,Kartu Kredit,Samarinda,3.0,2026-07-14,False
507,TRX00349,"drg. Salimah Usamah, S.Ked",Maiores basic,Olahraga,NaN,3,2026-06-30,Kartu Kredit,Batam,5.0,2026-06-30,False


3. Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang
sudah bersih.

In [ ]:
jumlah_per_kategori = df['category'].value_counts()
print(jumlah_per_kategori)

category
Kesehatan       96
Rumah Tangga    88
Buku            83
Elektronik      79
Olahraga        78
Fashion         66
Name: count, dtype: Int64
